[Relational inductive biases, deep learning, and graph networks](https://arxiv.org/abs/1806.01261)

GN framework can be seen as a generalization of many existing GNN architectures

| Traditional GNN | Mapping to `gn_block` Components                                          |
|-----------------|---------------------------------------------------------------------------|
| **GCN**         | Use only `ϕ_v` with simple aggregation (sum/mean).                        |
| **GAT**         | Use `ϕ_v` with attention-weighted aggregation (attention as edge importance). |

<br>

<img src="image/gn_blocks.png" width =500px>

1. $u \in \R^{d_u}$ is a global, graph-level attribute
2. $V \in \R^{n \times d_n}$ : a matrix with each row representing a vector of node-level attributes for one node
3. $E \in \R^{m \times d_m}$: a matrix with each row representing a vector of edge-level attributes for one edge

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.data import Data

class FullGNBlock(MessagePassing):
    '''
    1. edge_in_dim: d_m 
    2. node_in_dim: d_n
    3. global_in_dim: d_u
    '''
    def __init__(self, edge_in_dim, node_in_dim, global_in_dim, edge_out_dim, node_out_dim, global_out_dim):
        '''
        Define the edge update function (MLP)
        Input matrix size: [num_edges(m), edge_in_dim(d_m) + 2 * node_in_dim(d_n) + global_in_dim(d_u)]
        Output matrix size: [num_edges(m), edge_out_dim]
        '''
        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_in_dim + 2 * node_in_dim + global_in_dim, edge_out_dim),
            nn.ReLU(),
            nn.Linear(edge_out_dim, edge_out_dim)
        )

        '''
        Define the node update function (MLP)
        Input matrix size: [num_nodes(n), node_in_dim(d_n) + edge_out_dim + global_in_dim]
        Output matrix size: [num_nodes(n), node_out_dim]
        '''
        self.node_mlp = nn.Sequential(
            nn.Linear(node_in_dim + edge_out_dim + global_in_dim, node_out_dim),
            nn.ReLU(),
            nn.Linear(node_out_dim, node_out_dim)
        )

        '''
        Define the global update function (MLP)
        Input matrix size: [1, global_in_dim + node_out_dim + edge_out_dim]
        Output matrix size: [1, global_out_dim]
        '''
        self.global_mlp = nn.Sequential(
            nn.Linear(global_in_dim + node_out_dim + edge_out_dim, global_out_dim),
            nn.ReLU(),
            nn.Linear(global_out_dim, global_out_dim)
        )

    def forward(self, x, edge_index, edge_attr, u):
        # x: Node features [num_nodes, node_in_dim]
        # edge_index: Edge connectivity [2, num_edges]
        # edge_attr: Edge features [num_edges, edge_in_dim]
        # u: Global features [1, global_in_dim]

        # Step 1: Update Edge Features
        row, col = edge_index
        edge_inputs = torch.cat([x[row], x[col], edge_attr, u.repeat(edge_attr.size(0), 1)], dim=-1)
        edge_attr = self.edge_mlp(edge_inputs)

        # Step 2: Update Node Features
        '''
        Aggregated_edge_attr: edge features (edge_attr) are aggregated at each node
        1. sum over the incoming edges for each node
        2. dim: [num_nodes, edge_out_dim]

        node_inputs
        1. dim: [num_nodes, node_in_dim + edge_out_dim + global_in_dim]
        '''
        aggregated_edge_attr = self.propagate(edge_index, x=x, edge_attr=edge_attr)  # Aggregate updated edge features
        node_inputs = torch.cat([x, aggregated_edge_attr, u.repeat(x.size(0), 1)], dim=-1)
        x = self.node_mlp(node_inputs)

        # Step 3: Update Global Features
        '''
        global_inputs
        1. dim: [1, global_in_dim + node_out_dim + edge_out_dim]

        x.mean(...): [num_nodes, node_out_dim] -> [1, node_out_dim]
        edge_attr.mean(...): mean of edge features -> [1, edge_out_dim]
        '''
        global_inputs = torch.cat([u, x.mean(dim=0, keepdim=True), edge_attr.mean(dim=0, keepdim=True)], dim=-1)
        u = self.global_mlp(global_inputs)

        return x, edge_attr, u


# Example Usage:
# Create a small example graph with 3 nodes, 2 edges, and some dummy features.
num_nodes = 3
num_edges = 2
node_features = torch.rand((num_nodes, 5))  # 5-dimensional node features
edge_features = torch.rand((num_edges, 4))  # 4-dimensional edge features
global_features = torch.rand((1, 6))        # 6-dimensional global features

# Edge index tensor, shape [2, num_edges]
edge_index = torch.tensor([[0, 1],
                           [1, 2]], dtype=torch.long)

# Define the GN block
gn_block = FullGNBlock(edge_in_dim=4, node_in_dim=5, global_in_dim=6, 
                       edge_out_dim=4, node_out_dim=5, global_out_dim=6)

# Forward pass through the GN block
updated_nodes, updated_edges, updated_global = gn_block(node_features, edge_index, edge_features, global_features)

print("Updated Node Features:\n", updated_nodes)
print("Updated Edge Features:\n", updated_edges)
print("Updated Global Features:\n", updated_global)


1. Edge inputs: 

    <img src="image/edge_input.JPG" width=500px>

2. Aggregated edge attribute

    <img src="image/edge_attr.JPG" width=500px>

3. Node inputs:

    <img src="image/node_input.JPG" width=500px>